# 01 — Data pipeline
Download RTLCoder + MG-Verilog, download eval sets (eval-only, never trained on), build the corpus (normalise -> dedup -> contamination check -> tag -> split).

In [ ]:

# --- Self-contained Colab bootstrap (Part 0) ---
# Every notebook does this independently: Colab does not guarantee a new
# notebook tab reuses a previous notebook's VM, so nothing installed or
# cloned in another notebook can be assumed to exist here. This is
# idempotent -- re-running it (e.g. because you ARE still on the same
# runtime) just no-ops the clone and re-pulls latest.
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

REPO = "https://github.com/saiswaroop25-pixel/verilog-slm"
if not os.path.exists('/content/verilog-slm'):
    !git clone {REPO} /content/verilog-slm
%cd /content/verilog-slm
!git pull

CKPT = '/content/drive/MyDrive/verilog-slm/checkpoints'
LOGS = '/content/drive/MyDrive/verilog-slm/logs'
os.makedirs(CKPT, exist_ok=True); os.makedirs(LOGS, exist_ok=True)
!ln -sfn {CKPT} artifacts_drive_ckpt
!ln -sfn {LOGS} artifacts_drive_logs

In [ ]:
# Pinned deps (Part 0 hygiene: Colab silently upgrades packages between sessions)
!pip install -q -r requirements.txt -r requirements-train.txt

In [ ]:
# requirements-train.txt deliberately doesn't pin torch (Colab ships one
# already matched to the VM's CUDA driver) -- log what's actually here
# instead, per Part 0's "pin every dependency" hygiene.
import torch
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU available:', torch.cuda.is_available())

In [ ]:
import os
# RTL toolchain: iverilog is required (compile+simulate). yosys and verible
# are optional -- the harness soft-gates on them (see docs/industry_standards.md)
# but install them here so the synthesis/lint stages actually run instead of
# being recorded as 'skipped'.
!apt-get -qq update && apt-get -qq install -y iverilog yosys > /dev/null

# verible's release asset filename embeds a version string that changes
# every release (verible-v0.0-NNNN-gHASH-linux-static-x86_64.tar.gz), so a
# fixed "latest/download/<literal-name>" URL goes stale -- resolve the
# actual asset URL via the GitHub API instead. Chained as one shell
# command (not separate `!` lines) so the VERIBLE_URL variable survives
# across the pipe/curl/tar steps -- each `!` line is its own subprocess,
# so a bare shell variable assignment on its own line would silently be
# treated as Python and never reach bash at all.
!VERIBLE_URL=$(curl -s https://api.github.com/repos/chipsalliance/verible/releases/latest \
  | grep -o '"browser_download_url": *"[^"]*linux-static-x86_64.tar.gz"' \
  | head -1 | cut -d'"' -f4) && echo "resolved verible URL: $VERIBLE_URL" && \
  curl -sL "$VERIBLE_URL" -o /tmp/verible.tar.gz && \
  mkdir -p /opt/verible && tar -xzf /tmp/verible.tar.gz -C /opt/verible --strip-components=1

os.environ['PATH'] += ':/opt/verible/bin'
!iverilog -V | head -1
!yosys -V
!verible-verilog-lint --version

In [ ]:
# Record the GPU model at the start of every run -- required for the
# per-GPU-hour metric (Part 0 non-negotiable hygiene) to mean anything.
!nvidia-smi -L

In [ ]:
import os
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/eval', exist_ok=True)

# Fetch RTLCoder (27k+ pairs) and MG-Verilog (11k+ modules x 3 description
# levels) via the `datasets` library, then re-shape each into this repo's
# {instruction, code} jsonl schema before build_corpus.py normalises it further.
!pip install -q datasets
from datasets import load_dataset
import json

rtlcoder = load_dataset("ishorn5/RTLCoder-v1.1", split="train")
with open('data/raw/rtlcoder.jsonl', 'w') as f:
    for row in rtlcoder:
        f.write(json.dumps({"instruction": row.get("Instruction", row.get("instruction")),
                             "code": row.get("Response", row.get("code"))}) + "\n")

mg_verilog = load_dataset("GaTech-EIC/MG-Verilog", split="train")
with open('data/raw/mg_verilog.jsonl', 'w') as f:
    for row in mg_verilog:
        # MG-Verilog ships 3 description granularities per module; use the
        # 'detailed' level as the instruction so difficulty roughly matches RTLCoder
        f.write(json.dumps({"instruction": row.get("description_detailed", row.get("description")),
                             "code": row.get("code")}) + "\n")

print('RTLCoder:', sum(1 for _ in open('data/raw/rtlcoder.jsonl')))
print('MG-Verilog:', sum(1 for _ in open('data/raw/mg_verilog.jsonl')))

In [ ]:
# Eval sets -- EVAL ONLY, never in the training corpus. VerilogEval v2 (156
# problems) and RTLLM v2 (50 designs). Each row needs: id, instruction,
# testbench, top_module, tier (assigned via the tagger against the
# reference solution), code (reference solution -- used only for the
# contamination check, never for scoring).
!git clone --depth 1 https://github.com/NVlabs/verilog-eval /tmp/verilogeval
!git clone --depth 1 https://github.com/hkust-zhiyao/RTLLM /tmp/rtllm
# scripts/build_eval_jsonl.py's GLOB_PATTERNS must match whatever tag you
# just cloned -- both repos have reorganised their layout before, so fix
# the patterns there first if this step reports 0 problem directories.
!python -m scripts.build_eval_jsonl --src /tmp/verilogeval --out data/eval/verilogeval_v2.jsonl --benchmark verilogeval
!python -m scripts.build_eval_jsonl --src /tmp/rtllm --out data/eval/rtllm_v2.jsonl --benchmark rtllm

In [ ]:
!python -m src.data.build_corpus \
  --sources rtlcoder=data/raw/rtlcoder.jsonl mg-verilog=data/raw/mg_verilog.jsonl \
  --eval-sets data/eval/verilogeval_v2.jsonl data/eval/rtllm_v2.jsonl \
  --out artifacts/corpus.jsonl \
  --probe-frac 0.10 --seed 1337

In [ ]:
# Sanity-check tier/split balance before training on it
import json
from collections import Counter
rows = [json.loads(l) for l in open('artifacts/corpus.jsonl')]
print('total:', len(rows))
print('by split:', Counter(r['split'] for r in rows))
print('by tier (train):', Counter(r['tags']['tier'] for r in rows if r['split']=='train'))
print('by tier (probe):', Counter(r['tags']['tier'] for r in rows if r['split']=='probe'))